In [ ]:
import requests
from bs4 import BeautifulSoup
import re

def scrape_wikipedia_page(url: str) -> dict:

    headers = {
        'User-Agent': 'DataScienceRAGProject/1.0 (Student Project)'
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        raise Exception(f"eroare {url}: code {response.status_code}")
    
    soup = BeautifulSoup(response.text, 'html.parser')
    title = soup.find('h1', id='firstHeading').text
    content_div = soup.find('div', id='bodyContent')
    for element in content_div.find_all(['table', 'style', 'script', 'sup', 'div'], class_=['reflist', 'navbox', 'sidebar', 'mw-editsection']):
        element.decompose()
        
    paragraphs = content_div.find_all('p')
    clean_text = []
    
    for p in paragraphs:
        text = p.get_text().strip()
        if len(text) > 40:
            text = re.sub(r'\[\d+\]', '', text)
            clean_text.append(text)
            
    full_text = "\n\n".join(clean_text)
    
    return {
        "title": title,
        "url": url,
        "text": full_text
    }

urls_to_scrape = [
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
    "https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)",
    "https://en.wikipedia.org/wiki/Prompt_engineering",
    "https://en.wikipedia.org/wiki/Large_language_model",
    "https://en.wikipedia.org/wiki/Vector_database",
    "https://en.wikipedia.org/wiki/Word_embedding",
    "https://en.wikipedia.org/wiki/Stock_market",
    "https://en.wikipedia.org/wiki/Gastronomy",
    "https://en.wikipedia.org/wiki/Mediterranean_diet",
    "https://en.wikipedia.org/wiki/Coffee",
    "https://en.wikipedia.org/wiki/Olympic_Games",
    "https://en.wikipedia.org/wiki/Association_football",
    "https://en.wikipedia.org/wiki/Aerobic_exercise",
]

scraped_documents = []
for url in urls_to_scrape:
    data = scrape_wikipedia_page(url)
    scraped_documents.append(data)
    print(f"✅ : {data['title']} ({len(data['text'])} caractere)")

✅ : Retrieval-augmented generation (8798 caractere)
✅ : Transformer (deep learning) (63558 caractere)
✅ : Prompt engineering (16875 caractere)
✅ : Large language model (49105 caractere)
✅ : Vector database (2938 caractere)
✅ : Word embedding (9687 caractere)
✅ : Stock market (36758 caractere)
✅ : Gastronomy (4690 caractere)
✅ : Mediterranean diet (15516 caractere)
✅ : Coffee (51588 caractere)
✅ : Olympic Games (83719 caractere)
✅ : Association football (42952 caractere)
✅ : Aerobic exercise (7891 caractere)


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_sizes = {
    "small": 200,
    "medium": 500,
    "large": 1000
}
chunk_overlap = 50
chunked_datasets = {}

for size_name, size_val in chunk_sizes.items():
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=size_val,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    ) 
    all_chunks = []
    for doc in scraped_documents:
        chunks = text_splitter.split_text(doc["text"])
        
        for i, chunk_text in enumerate(chunks):
            all_chunks.append({
                "chunk_id": f"{doc['title']}_{size_name}_{i}",
                "text": chunk_text,
                "source": doc["title"],
                "url": doc["url"],
                "size_category": size_name
            })
            
    chunked_datasets[size_name] = all_chunks
    print(f" Config [{size_name.upper()}] (size={size_val}): {len(all_chunks)} chunks generate")

print("\nCHUNK Medium")
print("Text:", chunked_datasets["medium"][0]["text"])
print("lung carac:", len(chunked_datasets["medium"][0]["text"]))

 Config [SMALL] (size=200): 2763 chunks generate
 Config [MEDIUM] (size=500): 1092 chunks generate
 Config [LARGE] (size=1000): 544 chunks generate

CHUNK Medium
Text: Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this enables LLM-based chatbots to
lung carac: 496


In [4]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

print("se incarca...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_stores = {}

for size_name, chunks in chunked_datasets.items():
    print(f"indexare vectori [{size_name.upper()}] ({len(chunks)} chunks)...")
    texts = [c["text"] for c in chunks]
    metadatas = [{"source": c["source"], "url": c["url"], "size_category": c["size_category"]} for c in chunks]
    ids = [c["chunk_id"] for c in chunks]
    
    db = Chroma.from_texts(
        texts=texts,
        embedding=embedding_model,
        metadatas=metadatas,
        ids=ids,
        collection_name=f"rag_collection_{size_name}",
        persist_directory="./chroma_db_medium"
    )
    
    vector_stores[size_name] = db
    print(f"bd vectoriala [{size_name.upper()}] e gata")

C:\Users\User1\AppData\Local\Temp\ipykernel_5900\2939596145.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


se incarca...


C:\Users\User1\AppData\Local\Temp\ipykernel_5900\2939596145.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

indexare vectori [SMALL] (2763 chunks)...
bd vectoriala [SMALL] e gata
indexare vectori [MEDIUM] (1092 chunks)...
bd vectoriala [MEDIUM] e gata
indexare vectori [LARGE] (544 chunks)...
bd vectoriala [LARGE] e gata


In [5]:
import numpy as np

query = "What is Retrieval-Augmented Generation and how does it work?"

print(f"intrebare de test: '{query}'")

for size_name, db in vector_stores.items():
    print(f"\n rezultate pentru configuratia [{size_name.upper()}]:")
    results = db.similarity_search_with_score(query, k=2)
    
    for i, (doc, dist_score) in enumerate(results, start=1):
        sim_score = 1 - dist_score
        
        print(f"Locul {i}:")
        print(f" • Distanță ChromaDB : {dist_score:.4f} ")
        print(f" • Similaritate Cos  : {sim_score:.4f} | {sim_score*100:.1f}%)")
        print(f" • Sursa            : {doc.metadata['source']}")
        print(f" • URL               : {doc.metadata['url']}")
        print(f" • Text Fragment     : {doc.page_content[:140]}...\n")

print("Similaritate Cosinus ")
query_vector = np.array(embedding_model.embed_query(query))

for size_name in ["small", "medium", "large"]:
    sample_text = chunked_datasets[size_name][0]["text"]
    doc_vector = np.array(embedding_model.embed_query(sample_text))
    
    cosine_sim = np.dot(query_vector, doc_vector) / (np.linalg.norm(query_vector) * np.linalg.norm(doc_vector))
    
    print(f"• Esantion [{size_name.upper():<6}] -> Similaritate Cos matematica: {cosine_sim:.4f}")


intrebare de test: 'What is Retrieval-Augmented Generation and how does it work?'

 rezultate pentru configuratia [SMALL]:
Locul 1:
 • Distanță ChromaDB : 0.4809 
 • Similaritate Cos  : 0.5191 | 51.9%)
 • Sursa            : Retrieval-augmented generation
 • URL               : https://en.wikipedia.org/wiki/Retrieval-augmented_generation
 • Text Fragment     : Retrieval-augmented generation is used in applications where generated responses need to be grounded in external or frequently updated infor...

Locul 2:
 • Distanță ChromaDB : 0.5540 
 • Similaritate Cos  : 0.4460 | 44.6%)
 • Sursa            : Prompt engineering
 • URL               : https://en.wikipedia.org/wiki/Prompt_engineering
 • Text Fragment     : Retrieval-augmented generation is a technique that enables GenAI models to retrieve and incorporate new information. It modifies interaction...


 rezultate pentru configuratia [MEDIUM]:
Locul 1:
 • Distanță ChromaDB : 0.4956 
 • Similaritate Cos  : 0.5044 | 50.4%)
 • Sursa    

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

query = "What is Retrieval-Augmented Generation and how does it work?"
best_docs = vector_stores["small"].similarity_search(query, k=2)

context_text = "\n".join([f"- {doc.page_content}" for doc in best_docs])
sources_used = [doc.metadata['url'] for doc in best_docs]

prompt_input = f"""Answer the question based strictly on the context provided below.

CONTEXT:
{context_text}

QUESTION:
{query}

ANSWER:"""

print("Se incarca modelul LLM local (google/flan-t5-base)...")

model_id = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

inputs = tokenizer(prompt_input, return_tensors="pt", truncation=True, max_length=512)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)


print("\n RASPUNS GENERAT LOCAL:")
print(generated_text)
print(" SURSE UTILIZATE DIN CHROMADB:")
for url in set(sources_used):
    print(f"  • {url}")

Se incarca modelul LLM local (google/flan-t5-base)...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



 RASPUNS GENERAT LOCAL:
Retrieval-augmented generation is a technique that enables GenAI models to retrieve and incorporate new information.[citation needed]
🔗 SURSE UTILIZATE DIN CHROMADB:
  • https://en.wikipedia.org/wiki/Retrieval-augmented_generation
  • https://en.wikipedia.org/wiki/Prompt_engineering


In [ ]:

print("FAILURE CASE ANALYSIS")
out_of_domain_query = "What is the capital city of Australia and what is its population?"

irrelevant_docs = vector_stores["medium"].similarity_search(out_of_domain_query, k=2)
irrelevant_context = "\n".join([f"- {doc.page_content}" for doc in irrelevant_docs])

failure_prompt = f"""Answer the question based strictly on the context provided below. If the information is not in the context, say "I don't know"

CONTEXT:
{irrelevant_context}

QUESTION:
{out_of_domain_query}

ANSWER:"""

inputs_fail = tokenizer(failure_prompt, return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    outputs_fail = model.generate(**inputs_fail, max_new_tokens=100)

print(f"intrebare : {out_of_domain_query}")
print(f"raspuns RAG: {tokenizer.decode(outputs_fail[0], skip_special_tokens=True)}")

print("\nHALLUCINATION EXAMPLE")
hallucination_query = "Who invented Retrieval-Augmented Generation in 1925 at Harvard?"
unconstrained_prompt = f"Question: {hallucination_query}\nAnswer in detail:"

inputs_hal = tokenizer(unconstrained_prompt, return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    outputs_hal = model.generate(**inputs_hal, max_new_tokens=100)

print(f"intrebare halucinare: {hallucination_query}")
print(f"raspuns LLM fara RAG: {tokenizer.decode(outputs_hal[0], skip_special_tokens=True)}")

FAILURE CASE ANALYSIS
intrebare : What is the capital city of Australia and what is its population?
raspuns RAG: I don't know

HALLUCINATION EXAMPLE
intrebare halucinare: Who invented Retrieval-Augmented Generation in 1925 at Harvard?
raspuns LLM fara RAG: john d. salinger
